In [1]:
"""Self-contained Kaggle notebook script: PatchCore on GoodsAD food_box, with
brightness/contrast augmentation.

Not meant to be run as one file — copy each "# %% CELL" section into its own
Kaggle notebook cell, in order. Written to have NO dependency on this
project's Claude Code session: it only needs the GoodsAD food_box dataset
(as a Kaggle Dataset input) and internet access turned on (for pip installs
and the pretrained backbone weights download).

Setup before running:
  1. Upload the GoodsAD food_box dataset zip (the same ~1.7GB zip already
     used earlier in this project) as a new Kaggle Dataset. Kaggle will
     auto-extract it. Note the dataset's mount path, usually
     /kaggle/input/<your-dataset-name>/.
  2. Create a new Kaggle Notebook, attach that dataset, and under
     Settings -> turn ON internet access.
  3. Edit DATA_ROOT below (Cell 1) to match your dataset's actual mount path
     and folder layout (it should contain train/good, test/<defect_type>/,
     ground_truth/<defect_type>/ for the food_box category).
  4. Run cells in order. Expect ~1-1.5 hours on Kaggle's CPU, faster if you
     select a GPU runtime (not required -- PatchCore does not need one for
     a dataset this size, but it won't hurt).

Baseline to compare against (same dataset, no augmentation, trained
elsewhere): image_AUROC 0.8027, image_F1Score 0.8000, pixel_AUROC 0.9746,
pixel_F1Score 0.3482.
"""

'Self-contained Kaggle notebook script: PatchCore on GoodsAD food_box, with\nbrightness/contrast augmentation.\n\nNot meant to be run as one file — copy each "# %% CELL" section into its own\nKaggle notebook cell, in order. Written to have NO dependency on this\nproject\'s Claude Code session: it only needs the GoodsAD food_box dataset\n(as a Kaggle Dataset input) and internet access turned on (for pip installs\nand the pretrained backbone weights download).\n\nSetup before running:\n  1. Upload the GoodsAD food_box dataset zip (the same ~1.7GB zip already\n     used earlier in this project) as a new Kaggle Dataset. Kaggle will\n     auto-extract it. Note the dataset\'s mount path, usually\n     /kaggle/input/<your-dataset-name>/.\n  2. Create a new Kaggle Notebook, attach that dataset, and under\n     Settings -> turn ON internet access.\n  3. Edit DATA_ROOT below (Cell 1) to match your dataset\'s actual mount path\n     and folder layout (it should contain train/good, test/<defect_ty

In [2]:
import os
from pathlib import Path

# Path to the extracted GoodsAD food_box folder as Kaggle mounted it.
# Must contain: train/good, test/good, test/<defect_type>/, ground_truth/<defect_type>/
DATA_ROOT = Path("/kaggle/input/datasets/dtcrxs/goodsad/food_box/food_box")
# Kaggle's /kaggle/input is read-only, so build the merged (flat
# abnormal/masks) layout under /kaggle/working instead.
MERGED_ROOT = Path("/kaggle/working/GoodsAD_merged/food_box")

RESULTS_DIR = Path("/kaggle/working/results")

In [3]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "anomalib[core]==2.6.0"],
    check=True,
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 1.8 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.2/800.2 kB 21.6 MB/s eta 0:00:00


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'anomalib[core]==2.6.0'], returncode=0)

In [4]:
# GoodsAD structure (same workaround used earlier for the anomalib
# make_mvtec_ad_dataset pandas bug -- using the Folder datamodule sidesteps
# it entirely, so this step is required either way)
import shutil

abnormal_dir = MERGED_ROOT / "abnormal"
masks_dir = MERGED_ROOT / "masks"
abnormal_dir.mkdir(parents=True, exist_ok=True)
masks_dir.mkdir(parents=True, exist_ok=True)

defect_types = ["deformation", "opened", "surface_damage"]
copied = 0
for defect_type in defect_types:
    src_test_dir = DATA_ROOT / "test" / defect_type
    src_mask_dir = DATA_ROOT / "ground_truth" / defect_type
    if not src_test_dir.is_dir():
        print(f"Skipping missing defect type folder: {src_test_dir}")
        continue
    for img_path in sorted(src_test_dir.iterdir()):
        dest_name = f"{defect_type}_{img_path.name}"
        shutil.copy2(img_path, abnormal_dir / dest_name)
        mask_path = src_mask_dir / img_path.name
        if not mask_path.is_file():
            # GoodsAD masks are typically .png even when images are .jpg
            mask_path = src_mask_dir / (img_path.stem + ".png")
        if mask_path.is_file():
            dest_mask_name = f"{defect_type}_{img_path.stem}.png"
            shutil.copy2(mask_path, masks_dir / dest_mask_name)
        copied += 1

print(f"Copied {copied} abnormal images into {abnormal_dir}")
print(f"Abnormal: {len(list(abnormal_dir.iterdir()))} files, "
      f"Masks: {len(list(masks_dir.iterdir()))} files")

Copied 251 abnormal images into /kaggle/working/GoodsAD_merged/food_box/abnormal
Abnormal: 251 files, Masks: 251 files


In [5]:
# wide_resnet50_2 weights used throughout this project, from their original
# GitHub release, and point timm at the local file (works the same as this
# project's src/backbone_cache.py, inlined here since Kaggle notebooks don't
# have access to this repo's other files)
import urllib.request

WEIGHTS_DIR = Path("/kaggle/working/weights")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_URL = (
    "https://github.com/rwightman/pytorch-image-models/releases/"
    "download/v0.1-weights/wide_resnet50_racm-8234f177.pth"
)
WEIGHTS_PATH = WEIGHTS_DIR / "wide_resnet50_racm-8234f177.pth"

if not WEIGHTS_PATH.exists():
    print(f"Downloading backbone weights to {WEIGHTS_PATH} ...")
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)

import timm

_original_create_model = timm.create_model


def _patched_create_model(model_name, *args, **kwargs):
    if kwargs.get("pretrained") and model_name == "wide_resnet50_2":
        kwargs.pop("pretrained_cfg", None)
        kwargs["pretrained_cfg_overlay"] = {"file": str(WEIGHTS_PATH)}
    return _original_create_model(model_name, *args, **kwargs)


timm.create_model = _patched_create_model

In [6]:
# (matches this project's src/train_folder.py --augment flag, inlined)
from anomalib.data import Folder
from anomalib.engine import Engine
from anomalib.models import Patchcore
from torchvision.transforms import v2

# Kaggle/Colab notebooks can hit a RecursionError from tqdm's progress bar
# fighting with rich/ipykernel's Jupyter display hooks. We don't need the
# live bar, just the final metrics, so disable it outright.
import tqdm as tqdm_module

_original_tqdm_init = tqdm_module.tqdm.__init__


def _patched_tqdm_init(self, *args, **kwargs):
    kwargs["disable"] = True
    _original_tqdm_init(self, *args, **kwargs)


tqdm_module.tqdm.__init__ = _patched_tqdm_init

train_augmentations = v2.ColorJitter(brightness=0.2, contrast=0.2)

datamodule = Folder(
    name="food_box_augmented",
    normal_dir=DATA_ROOT / "train" / "good",
    abnormal_dir=abnormal_dir,
    mask_dir=masks_dir,
    normal_test_dir=DATA_ROOT / "test" / "good",
    train_batch_size=32,
    eval_batch_size=32,
    num_workers=2,
    train_augmentations=train_augmentations,
)

model = Patchcore(
    backbone="wide_resnet50_2",
    layers=["layer2", "layer3"],
    pre_trained=True,
    coreset_sampling_ratio=0.1,  # matches the no-augmentation baseline for a fair comparison
)

engine = Engine(default_root_dir=str(RESULTS_DIR))

print("Training PatchCore (augmented) on food_box ...")
test_results = engine.train(model=model, datamodule=datamodule)

print("\n=== Test results (augmented) ===")
for result in test_results:
    for key, value in result.items():
        formatted = f"{value:.4f}" if isinstance(value, float) else str(value)
        print(f"{key}: {formatted}")

print("\n=== Compare against baseline (no augmentation) ===")
print("image_AUROC:   0.8027")
print("image_F1Score: 0.8000")
print("pixel_AUROC:   0.9746")
print("pixel_F1Score: 0.3482")

ckpt_path = engine.trainer.checkpoint_callback.best_model_path
print(f"\nCheckpoint saved to: {ckpt_path}")

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.13/dist-packages/anomalib/models/image/dinomaly/components/layers.py:22: FutureWarning: The anomalib.models.components.dinov2 package is deprecated and will be removed in a future release. Please use the timm-based feature extractor instead: anomalib.models.components.feature_extractors.TimmFeatureExtractor
  from anomalib.models.components.dinov2.layers import Attention, DropPath, LayerScale, MemEffAttention
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning E

Training PatchCore (augmented) on food_box ...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.13/dist-packages/lightning/pytorch/core/optimizer.py:183: `LightningModule.configure_optimizers` returned `None`, this fit will run with no optimizer


┏━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name           ┃ Type           ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ pre_processor  │ PreProcessor   │      0 │ train │     0 │
│ 1 │ post_processor │ PostProcessor  │      0 │ train │     0 │
│ 2 │ evaluator      │ Evaluator      │      0 │ train │     0 │
│ 3 │ model          │ PatchcoreModel │ 24.9 M │ train │     0 │
└───┴────────────────┴────────────────┴────────┴───────┴───────┘

Trainable params: 24.9 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 24.9 M                                                                                               
Total estimated model params size (MB): 99.450                                                                     
Modules in train mode: 19                                                                                          
Modules in eval mode: 174                                                                                          
Total FLOPs: 0

...

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/loops/fit_loop.py:538: Found 174 module(s) in eval mode at the start of training. This may lead to unexpected behavior during training. If this is intentional, you can ignore this warning.
`Trainer.fit` stopped: `max_epochs=1` reached.


The following callbacks returned in `LightningModule.configure_callbacks` will override existing callbacks passed to Trainer: Evaluator, ImageVisualizer, PostProcessor, PreProcessor
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        image_AUROC        │    0.7700586915016174     │
│       image_F1Score       │    0.7985865473747253     │
│        pixel_AUROC        │    0.9761254787445068     │
│       pixel_F1Score       │    0.37118279933929443    │
└───────────────────────────┴───────────────────────────┘


=== Test results (augmented) ===
image_AUROC: 0.7701
image_F1Score: 0.7986
pixel_AUROC: 0.9761
pixel_F1Score: 0.3712

=== Compare against baseline (no augmentation) ===
image_AUROC:   0.8027
image_F1Score: 0.8000
pixel_AUROC:   0.9746
pixel_F1Score: 0.3482

Checkpoint saved to: /kaggle/working/results/Patchcore/food_box_augmented/v0/weights/lightning/model.ckpt
